In [1]:
import os

print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : /Users/kanellis/json-api-geocodage/notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['restaurants.json']


In [2]:
import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["ville"], "-", data["pays"])
print("nombre de restaurants :", len(data["restaurants"]))

<class 'dict'>
Catane - Italie
nombre de restaurants : 4


In [3]:
print(data["restaurants"][0]["nom"])
print(data["restaurants"][0]["notes"][0])

Gelateria Zio Pietro dal 1964
4.6


In [4]:
for r in data["restaurants"]:
    moyenne = sum(r["notes"]) / len(r["notes"])
    print(f"{r['nom']:32} {moyenne:.2f}")

Gelateria Zio Pietro dal 1964    4.70
A Casa d'Amici                   4.45
La Bitta                         4.15
Steak House                      4.00


In [5]:
print(json.dumps(data["restaurants"][1], ensure_ascii=False, indent=2))


{
  "nom": "A Casa d'Amici",
  "adresse": "Via Fischetti, 14, 95131 Catania CT",
  "type": "trattoria",
  "notes": [
    4.4,
    4.5
  ]
}


In [6]:
import pandas as pd

df = pd.json_normalize(data)
print(df.shape)
print(df.columns.tolist())

(1, 4)
['ville', 'pays', 'releve', 'restaurants']


In [7]:
df = pd.json_normalize(data, record_path="restaurants")
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes  
0    glacier       [4.6, 4.8, 4.7]  
1  trattoria            [4.4, 4.5]  
2    poisson  [4.2, 4.0, 4.3, 4.1]  
3      grill            [3.9, 4.1]  


In [8]:
df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes   ville    pays  
0    glacier       [4.6, 4.8, 4.7]  Catane  Italie  
1  trattoria            [4.4, 4.5]  Catane  Italie  
2    poisson  [4.2, 4.0, 4.3, 4.1]  Catane  Italie  
3      grill            [3.9, 4.1]  Catane  Italie  


In [9]:
print(len(villes))

NameError: name 'villes' is not defined

In [10]:
for ville in data["villes"]:
    print(f"Ville : {ville['nom']} | Nombre de restaurants : {len(ville['restaurants'])}")

KeyError: 'villes'

In [11]:
import json

# Ouvrir et charger le fichier JSON situé dans le dossier data
with open("../data/villes.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Fichier JSON chargé avec succès !")

Fichier JSON chargé avec succès !


In [12]:
for ville in data["villes"]:
    print(f"Ville : {ville['nom']} | Nombre de restaurants : {len(ville['restaurants'])}")


Ville : Catane | Nombre de restaurants : 3
Ville : Palerme | Nombre de restaurants : 2
Ville : Syracuse | Nombre de restaurants : 2


In [13]:
import pandas as pd

df = pd.json_normalize(
    data,
    record_path=['villes', 'restaurants'],
    meta=[
        'pays',
        ['villes', 'nom'],
        ['villes', 'region'],
        ['villes', 'habitants']
    ]
)

# Nettoyage et renommage des colonnes issues des niveaux intermédiaires
df = df.rename(columns={
    'villes.nom': 'ville',
    'villes.region': 'région',
    'villes.habitants': 'habitants'
})

# Affichage des colonnes principales demandées
print(df[['nom', 'type', 'ville', 'région', 'habitants']])

                             nom       type     ville  région habitants
0  Gelateria Zio Pietro dal 1964    glacier    Catane  Sicile    298762
1                 A Casa d'Amici  trattoria    Catane  Sicile    298762
2                       La Bitta    poisson    Catane  Sicile    298762
3              Antica Focacceria  trattoria   Palerme  Sicile    630828
4               Ferro di Cavallo  trattoria   Palerme  Sicile    630828
5                    Don Camillo    poisson  Syracuse  Sicile    121605
6               Gelateria Bianca    glacier  Syracuse  Sicile    121605


In [14]:
# Calcul de la note moyenne
df['note_moyenne'] = df['notes'].apply(lambda x: round(sum(x) / len(x), 2))

# 1. Quelle ville a la meilleure note moyenne ?
meilleure_ville = df.groupby('ville')['note_moyenne'].mean().idxmax()
print(f"Ville avec la meilleure note moyenne : {meilleure_ville}")

# 2. Combien de restaurants de chaque type, tous lieux confondus ?
types_count = df['type'].value_counts()
print("\nNombre de restaurants par type :")
print(types_count)

Ville avec la meilleure note moyenne : Syracuse

Nombre de restaurants par type :
type
trattoria    3
glacier      2
poisson      2
Name: count, dtype: int64


In [15]:
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})

print(r.status_code)

/Users/kanellis/json-api-geocodage/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


200


In [16]:
reponse = r.json()
print(type(reponse))
print(reponse.keys())

<class 'dict'>
dict_keys(['type', 'features', 'query'])


In [17]:
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (prenom.nom@exemple.fr)"}
r = requests.get(url, params={"q": "Catane"}, headers=headers)

print(r.status_code)
print(r.json()["features"][0]["properties"]["label"])

200
Catane 31310 Montesquieu-Volvestre


In [1]:
import requests

def interroger(url, params, headers=None):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur :", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

/Users/kanellis/json-api-geocodage/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
print(interroger("https://api-adresse.data.gouv.fr/search/", {"q": "Lille", "limit": 1}) is not None)
print(interroger("https://api-adresse.data.gouv.fr/nimportequoi", {"q": "Lille"}))

True
Le serveur a répondu une erreur : 404 Client Error: Not Found for url: https://api-adresse.data.gouv.fr/nimportequoi?q=Lille
None


In [3]:
adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

In [4]:
import requests

adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

# ==========================================
# Étape 1 — Une seule, d'abord
# ==========================================
print("--- ÉTAPE 1 ---")
url_api = "https://api-adresse.data.gouv.fr/search/"
premiere_adresse = adresses[0]

# On envoie la requête à l'API
reponse = requests.get(url_api, params={"q": premiere_adresse})
print(f"Code de statut : {reponse.status_code}")

data = reponse.json()
# On vérifie qu'il y a bien des résultats
if data.get("features"):
    premier_resultat = data["features"][0]["properties"]
    print(f"Label : {premier_resultat.get('label')}")
    print(f"Score : {premier_resultat.get('score')}")
else:
    print("Aucun résultat trouvé.")


# ==========================================
# Étape 2 — Les cinq avec une boucle
# ==========================================
print("\n--- ÉTAPE 2 ---")
resultats_delegues = []

for adresse in adresses:
    # Paramètres de la requête pour chaque adresse
    params = {"q": adresse, "limit": 1}
    resp = requests.get(url_api, params=params)

    # Valeurs par défaut si l'adresse est introuvable
    label = None
    score = None
    postcode = None
    city = None

    if resp.status_code == 200:
        features = resp.json().get("features")
        if features:
            props = features[0]["properties"]
            label = props.get("label")
            score = props.get("score")
            postcode = props.get("postcode")
            city = props.get("city")

    # Construction du dictionnaire pour cette adresse
    dictionnaire_adresse = {
        "adresse_saisie": adresse,
        "label": label,
        "score": score,
        "postcode": postcode,
        "city": city,
    }

    resultats_delegues.append(dictionnaire_adresse)

# Affichage du résultat final pour vérifier
import pprint

pprint.pprint(resultats_delegues)

--- ÉTAPE 1 ---
Code de statut : 200
Label : 20 Avenue de Ségur 75007 Paris
Score : 0.9716281818181818

--- ÉTAPE 2 ---
[{'adresse_saisie': '20 avenue de Ségur 75007 Paris',
  'city': 'Paris',
  'label': '20 Avenue de Ségur 75007 Paris',
  'postcode': '75007',
  'score': 0.9716281818181818},
 {'adresse_saisie': 'Grand Place, Lille',
  'city': 'Lille',
  'label': 'Place de Gand 59800 Lille',
  'postcode': '59800',
  'score': 0.5682435064935065},
 {'adresse_saisie': '1 rue de la Paix, Paris',
  'city': 'Paris',
  'label': '1 Rue de la Paix 75002 Paris',
  'postcode': '75002',
  'score': 0.9640599999999998},
 {'adresse_saisie': 'place Bellecour, Lyon',
  'city': 'Lyon',
  'label': 'Place Bellecour 69002 Lyon',
  'postcode': '69002',
  'score': 0.9673672727272726},
 {'adresse_saisie': 'zzzzz qwerty 99999',
  'city': None,
  'label': None,
  'postcode': None,
  'score': None}]


In [5]:
import pandas as pd
import requests

adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

url_api = "https://api-adresse.data.gouv.fr/search/"
resultats_delegues = []

for adresse in adresses:
    params = {"q": adresse, "limit": 1}

    # Ajout du paramètre timeout (ex: 5 secondes) exigé par les critères
    try:
        resp = requests.get(url_api, params=params, timeout=5)
    except requests.exceptions.Timeout:
        print(f"La requête pour '{adresse}' a expiré (timeout).")
        continue

    label, score, postcode, city = None, None, None, None

    if resp.status_code == 200:
        features = resp.json().get("features")
        if features:
            props = features[0]["properties"]
            label = props.get("label")
            score = props.get("score")
            postcode = props.get("postcode")
            city = props.get("city")

    resultats_delegues.append(
        {
            "adresse_saisie": adresse,
            "label": label,
            "score": score,
            "postcode": postcode,
            "city": city,
        }
    )

# ==========================================
# Étape 3 — En DataFrame (5 lignes)
# ==========================================
df_adresses = pd.DataFrame(resultats_delegues)

print("\n--- DATAFRAME FINAL ---")
print(df_adresses)


--- DATAFRAME FINAL ---
                   adresse_saisie                           label     score  \
0  20 avenue de Ségur 75007 Paris  20 Avenue de Ségur 75007 Paris  0.971628   
1              Grand Place, Lille       Place de Gand 59800 Lille  0.568244   
2         1 rue de la Paix, Paris    1 Rue de la Paix 75002 Paris  0.964060   
3           place Bellecour, Lyon      Place Bellecour 69002 Lyon  0.967367   
4              zzzzz qwerty 99999                            None       NaN   

  postcode   city  
0    75007  Paris  
1    59800  Lille  
2    75002  Paris  
3    69002   Lyon  
4     None   None  
